# HI5312 — Getting Started with MIMIC-IV

## Health Informatics Data Analytics

This notebook introduces the basics of working with **MIMIC-IV in GitHub Codespaces**. You will learn how to:

- navigate the MIMIC-IV folders;
- understand the `hosp` and `icu` modules;
- inspect compressed CSV files;
- use CSV files with Python and `pandas`;
- connect to PostgreSQL with `psql`;
- run basic SQL queries;
- bring SQL query results into Python;
- connect MIMIC-IV structure to the HI5312 concepts of **data type, grain, keys, and relationships**.

> **Important:** Use only instructor-approved course datasets in Codespaces. Do not upload PHI, workplace data, research data, or other confidential information.


## 1. MIMIC-IV as a relational health database

MIMIC-IV is **not one spreadsheet**. It is a collection of related tables representing different parts of clinical care.

A useful mental model is:

**Patient → Hospital Admission → ICU Stay → Clinical Event**

Common identifiers include:

| Identifier | Meaning | Typical level |
|---|---|---|
| `subject_id` | Patient identifier | Patient |
| `hadm_id` | Hospital admission identifier | Hospital encounter |
| `stay_id` | ICU stay identifier | ICU encounter |
| `itemid` | Clinical item/concept identifier | Measurement/event |

A central question throughout this notebook is:

> **What does one row represent?**

That is the **grain** of the table.


## 2. Locate the MIMIC-IV files in Codespaces

In the HI5312 Codespaces environment, the MIMIC-IV demo is stored under the repository's `data/mimic-iv-demo` area, with separate `hosp` and `icu` folders.

The code below searches several likely locations automatically.


In [1]:
from pathlib import Path

candidate_roots = [
    Path("data/mimic-iv-demo"),
    Path("../data/mimic-iv-demo"),
    Path("/workspaces/hi5304-notebooks/data/mimic-iv-demo"),
    Path("mimic-iv-demo"),
]

MIMIC_ROOT = next((p.resolve() for p in candidate_roots if p.exists()), None)

if MIMIC_ROOT is None:
    raise FileNotFoundError(
        "Could not find the MIMIC-IV demo folder. "
        "Check the Explorer and update candidate_roots if your path is different."
    )

HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

print("MIMIC-IV root:", MIMIC_ROOT)
print("Hosp folder exists:", HOSP_DIR.exists())
print("ICU folder exists:", ICU_DIR.exists())


MIMIC-IV root: /workspaces/hi5304-notebooks/data/mimic-iv-demo
Hosp folder exists: True
ICU folder exists: True


## 3. Explore the `hosp` and `icu` folders

The **`hosp` module** contains hospital-wide EHR data such as patients, admissions, diagnoses, procedures, laboratory events, microbiology, medications, services, and transfers.

The **`icu` module** contains ICU-specific event data such as ICU stays, charted bedside observations, inputs, outputs, procedures, and item dictionaries.


In [2]:
def list_files(folder, limit=50):
    files = sorted(folder.glob("*"))
    print(f"{folder.name}: {len(files)} files")
    for f in files[:limit]:
        print(" -", f.name)

list_files(HOSP_DIR)
print()
list_files(ICU_DIR)


hosp: 22 files
 - admissions.csv
 - d_hcpcs.csv
 - d_icd_diagnoses.csv
 - d_icd_procedures.csv
 - d_labitems.csv
 - diagnoses_icd.csv
 - drgcodes.csv
 - emar.csv
 - emar_detail.csv
 - hcpcsevents.csv
 - labevents.csv
 - microbiologyevents.csv
 - omr.csv
 - patients.csv
 - pharmacy.csv
 - poe.csv
 - poe_detail.csv
 - prescriptions.csv
 - procedures_icd.csv
 - provider.csv
 - services.csv
 - transfers.csv

icu: 9 files
 - caregiver.csv
 - chartevents.csv
 - d_items.csv
 - datetimeevents.csv
 - icustays.csv
 - ingredientevents.csv
 - inputevents.csv
 - outputevents.csv
 - procedureevents.csv


## 4. Inspect a compressed CSV file

MIMIC-IV files are commonly stored as `.csv.gz` files. `pandas` can read these files directly without manually unzipping them.

We will begin with `patients.csv.gz`.


In [4]:
import pandas as pd

patients_file = HOSP_DIR / "patients.csv"

patients_preview = pd.read_csv(patients_file, nrows=5)

display(patients_preview)


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10014729,F,21,2125,2011 - 2013,NaN
1,10003400,F,72,2134,2011 - 2013,2137-09-02
2,10002428,F,80,2155,2011 - 2013,NaN
3,10032725,F,38,2143,2011 - 2013,2143-03-30
4,10027445,F,48,2142,2011 - 2013,2146-02-09


### Think about the data types

For each column, distinguish among three ideas:

1. **Storage type** — how the computer stores the value.
2. **Statistical type** — categorical, continuous, binary, temporal, etc.
3. **Semantic role** — identifier, measurement, code, timestamp, outcome, and so forth.

For example, `subject_id` may be stored as an integer, but analytically it is a **nominal identifier**, not a quantitative measurement.


In [5]:
patients_preview.dtypes


subject_id           int64
gender                 str
anchor_age           int64
anchor_year          int64
anchor_year_group      str
dod                    str
dtype: object

## 5. Read selected columns from a CSV file

Large health datasets can contain many rows and columns. A good analytic habit is to load only what you need.


In [6]:
patients = pd.read_csv(
    patients_file,
    usecols=["subject_id", "gender", "anchor_age"],
)

print("Rows:", len(patients))
patients.head()


Rows: 100


,subject_id,gender,anchor_age
0,10014729,F,21
1,10003400,F,72
2,10002428,F,80
3,10032725,F,38
4,10027445,F,48


## 6. Examine another table: admissions

The `patients` table is approximately **one row per patient**.

The `admissions` table is approximately **one row per hospital admission**.

Therefore, one patient may appear multiple times in `admissions`.


In [8]:
import pandas as pd

# Use the uncompressed CSV file
admissions_file = HOSP_DIR / "admissions.csv"

# Load the first 1,000 admission records
admissions = pd.read_csv(
    admissions_file,
    usecols=[
        "subject_id",
        "hadm_id",
        "admittime",
        "dischtime",
        "admission_type"
    ],
    nrows=1000
)

display(admissions.head())


,subject_id,hadm_id,admittime,dischtime,admission_type
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,URGENT
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,URGENT
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,URGENT
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,URGENT
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,URGENT


### Convert text to datetime

CSV files do not inherently preserve database data types. Dates and times may initially be read as text, so convert them explicitly before temporal analysis.


In [9]:
admissions["admittime"] = pd.to_datetime(admissions["admittime"])
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"])

admissions["length_of_stay_days"] = (
    admissions["dischtime"] - admissions["admittime"]
).dt.total_seconds() / 86400

admissions[
    ["subject_id", "hadm_id", "admittime", "dischtime", "length_of_stay_days"]
].head()


,subject_id,hadm_id,admittime,dischtime,length_of_stay_days
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,8.975000
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,7.841667
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,5.731250
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,1.736806
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,2.090972


## 7. Join tables in Python

The `patients` and `admissions` tables connect through `subject_id`.

Before joining, remember their grain:

- `patients`: one row per patient;
- `admissions`: one row per hospitalization.

This is a **one-to-many relationship**.


In [10]:
patient_admissions = admissions.merge(
    patients,
    on="subject_id",
    how="left",
)

patient_admissions.head()


,subject_id,hadm_id,admittime,dischtime,admission_type,length_of_stay_days,gender,anchor_age
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,URGENT,8.975000,M,47
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,URGENT,7.841667,M,58
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,URGENT,5.731250,M,79
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,URGENT,1.736806,M,52
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,URGENT,2.090972,F,82


In [11]:
patient_admissions = admissions.merge(
    patients,
    on="subject_id",
    how="left",
)

patient_admissions.head()

print("Admissions before join:", len(admissions))
print("Rows after join:", len(patient_admissions))
print("Unique patients:", patient_admissions["subject_id"].nunique())


Admissions before join: 275
Rows after join: 275
Unique patients: 100


## 8. Basic descriptive analysis in Python

Once the structure is understood and validated, begin with simple descriptive analysis.


In [12]:
patient_admissions.groupby("admission_type").agg(
    admissions=("hadm_id", "count"),
    mean_age=("anchor_age", "mean"),
    mean_length_of_stay=("length_of_stay_days", "mean"),
).sort_values("admissions", ascending=False)


,admissions,mean_age,mean_length_of_stay
admission_type,,,
EW EMER.,104,61.942308,7.279320
OBSERVATION ADMIT,45,59.222222,8.151821
URGENT,38,60.815789,9.850841
EU OBSERVATION,30,59.400000,0.916921
SURGICAL SAME DAY ADMISSION,18,62.333333,5.696258
DIRECT EMER.,15,58.533333,9.389676
ELECTIVE,13,59.846154,8.190759
DIRECT OBSERVATION,7,68.285714,1.445139
AMBULATORY OBSERVATION,5,66.200000,1.015278


# Part II — Using PostgreSQL and SQL

The Codespaces environment also includes a PostgreSQL database containing MIMIC-IV tables.

From a **Codespaces terminal**, connect with:

```bash
psql -h db -U hi5304_user -d hi5304
```

After entering the database password, you should see a prompt similar to:

```text
hi5304=#
```

Once you are inside `psql`, do **not** type the outer `psql -h ...` command again.


## 9. Useful `psql` navigation commands

Run these **inside `psql`**:

```sql
\dn
```

Lists schemas.

```sql
\dt mimiciv_hosp.*
```

Lists tables in the hospital schema.

```sql
\dt mimiciv_icu.*
```

Lists tables in the ICU schema.

```sql
\d mimiciv_hosp.admissions
```

Describes the columns and database data types in `admissions`.

```sql
\q
```

Exits `psql`.

If output pauses at `(END)`, press `q` to exit the pager.


## 10. Run a basic SQL query

```sql
SELECT *
FROM mimiciv_hosp.admissions
LIMIT 10;
```

A better production habit is to name only the columns you need:

```sql
SELECT
    subject_id,
    hadm_id,
    admittime,
    dischtime
FROM mimiciv_hosp.admissions
LIMIT 10;
```


## 11. Filter rows with `WHERE`

```sql
SELECT
    subject_id,
    hadm_id,
    admission_type,
    admittime
FROM mimiciv_hosp.admissions
WHERE admission_type = 'URGENT'
ORDER BY admittime
LIMIT 20;
```

Think of `WHERE` as the point where clinical inclusion and exclusion criteria become reproducible logic.


## 12. Join MIMIC-IV tables with SQL

```sql
SELECT
    p.subject_id,
    p.gender,
    p.anchor_age,
    a.hadm_id,
    a.admittime,
    a.dischtime
FROM mimiciv_hosp.patients AS p
JOIN mimiciv_hosp.admissions AS a
    ON p.subject_id = a.subject_id
LIMIT 20;
```

Before every join, ask:

- What does one row represent in each table?
- What key connects the tables?
- Is the relationship one-to-one, one-to-many, or many-to-many?
- Could the join multiply rows?


## 13. Aggregate data with SQL

```sql
SELECT
    admission_type,
    COUNT(*) AS n_admissions
FROM mimiciv_hosp.admissions
GROUP BY admission_type
ORDER BY n_admissions DESC;
```

For health analytics, always distinguish among:

- number of **rows**;
- number of **encounters**;
- number of **unique patients**.


## 14. Basic SQL quality checks

```sql
SELECT COUNT(*)
FROM mimiciv_hosp.admissions;
```

```sql
SELECT COUNT(DISTINCT subject_id)
FROM mimiciv_hosp.admissions;
```

```sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT subject_id) AS patients,
    COUNT(DISTINCT hadm_id) AS admissions
FROM mimiciv_hosp.admissions;
```

These checks help reveal table grain and protect against incorrect denominators.


# Part III — Bringing SQL Results into Python

A common health data science workflow is:

**PostgreSQL / SQL → analytic dataset → pandas → analysis → visualization**


## 15. Connect Python to PostgreSQL

Do **not** hard-code the database password in the notebook. The following cell asks for it securely at runtime.


In [ ]:
from getpass import getpass
from sqlalchemy import create_engine
from urllib.parse import quote_plus

DB_HOST = "db"
DB_NAME = "hi5304"
DB_USER = "hi5304_user"

DB_PASSWORD = getpass("Enter the PostgreSQL password: ")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}/{DB_NAME}"
)

print("Database engine created.")


## 16. Run SQL from Python

`pandas.read_sql_query()` executes SQL and returns the result as a DataFrame.


In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("PGUSER", "hi5304_user"),
    password=os.getenv("PGPASSWORD", "hi5304"),
    host=os.getenv("PGHOST", "db"),
    port=int(os.getenv("PGPORT", "5432")),
    database=os.getenv("PGDATABASE", "hi5304")
)

engine = create_engine(url, pool_pre_ping=True)

print("SQLAlchemy engine initialized.")

SQLAlchemy engine initialized.


In [5]:
import pandas as pd
from sqlalchemy import text

query = text("""
SELECT
    subject_id,
    hadm_id,
    admission_type,
    admittime,
    dischtime
FROM hi5312_demo.admissions
LIMIT 100;
""")

with engine.connect() as conn:
    admissions_sql = pd.read_sql_query(query, conn)

# Inspect the DataFrame
admissions_sql.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   subject_id      100 non-null    int64
 1   hadm_id         100 non-null    int64
 2   admission_type  100 non-null    str  
 3   admittime       100 non-null    str  
 4   dischtime       100 non-null    str  
dtypes: int64(2), str(3)
memory usage: 4.0 KB


## 17. Run an aggregate SQL query and analyze the result in Python


In [7]:
import pandas as pd
from sqlalchemy import text

query = text("""
SELECT
    admission_type,
    COUNT(*) AS n_admissions
FROM hi5312_demo.admissions
GROUP BY admission_type
ORDER BY n_admissions DESC;
""")

with engine.connect() as conn:
    admission_counts = pd.read_sql_query(query, conn)

display(admission_counts)


,admission_type,n_admissions
0,EW EMER.,104
1,OBSERVATION ADMIT,45
2,URGENT,38
3,EU OBSERVATION,30
4,SURGICAL SAME DAY ADMISSION,18
5,DIRECT EMER.,15
6,ELECTIVE,13
7,DIRECT OBSERVATION,7
8,AMBULATORY OBSERVATION,5


## 18. Compare CSV, SQL, and Python

### CSV files
Best for learning raw files, flat-file structure, and basic `pandas`.

### SQL / PostgreSQL
Best for large relational tables, filtering, joining, aggregation, and cohort construction.

### Python / pandas
Best for cleaning, exploratory analysis, feature engineering, statistics, modeling, and visualization.

These approaches work **together**, not in competition.


# Practice Exercises

### Exercise 1 — Navigation
1. List all files in `hosp`.
2. List all files in `icu`.
3. Identify three encounter-level tables and three repeated-event tables.

### Exercise 2 — CSV
1. Open the first 10 rows of `admissions.csv.gz`.
2. Display the column names.
3. Display the pandas data types.
4. Count unique `subject_id` values in the first 1,000 rows.

### Exercise 3 — SQL
1. List the MIMIC-IV schemas.
2. Describe `mimiciv_hosp.patients`.
3. Retrieve 10 rows from `mimiciv_hosp.patients`.
4. Count total admissions.
5. Count unique patients in admissions.

### Exercise 4 — Join
Join `patients` and `admissions` using `subject_id`.

Answer:
- What is the grain before the join?
- What is the grain after the join?
- Why can one patient appear more than once?

### Exercise 5 — Data types
Choose five MIMIC-IV variables and classify each by:
- database storage type;
- statistical data type;
- semantic role.


# Key Takeaways

1. **MIMIC-IV is a relational database, not a spreadsheet.**
2. The `hosp` and `icu` modules represent different parts of clinical care.
3. Always ask **what one row represents** before analyzing a table.
4. An integer can be an identifier rather than a quantitative variable.
5. `subject_id`, `hadm_id`, and `stay_id` are central relational keys.
6. CSV, SQL, and Python are complementary tools.
7. SQL is especially useful for filtering, joining, aggregating, and constructing cohorts.
8. Python is especially useful after the analytic dataset has been defined and validated.
9. Data quality checks belong throughout the workflow.
10. Understanding data structure comes **before** sophisticated analysis.

> **Health informatics principle:** trustworthy analysis begins with trustworthy structure.
